# Surf Maneuver Recognition: V-JEPA 2 Features

Each clip is described by the frozen V-JEPA 2 video encoder ([`facebook/vjepa2-vitl-fpc64-256`](https://huggingface.co/facebook/vjepa2-vitl-fpc64-256), MIT license).
The encoder runs once per clip and its output is cached, so the classifier on top can be trained many times at almost no cost.

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoVideoProcessor

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.inventory import DATA_DIR
from src.data.video import read_clip

splits = pd.read_csv(PROJECT_ROOT / "data" / "splits.csv")
print(len(splits), "clips")

c:\Users\fabri\Desktop\Portfolio\surf-maneuver-recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1116 23:50:09.048000 43412 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


1056 clips


## 2. Loading the encoder

In [2]:
MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"
device = "cuda"

processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
# float16 halves the memory use; the RTX 3050 has 6 GB
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

c:\Users\fabri\Desktop\Portfolio\surf-maneuver-recognition\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fabri\.cache\huggingface\hub\models--facebook--vjepa2-vitl-fpc64-256. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Users

326M parameters


## 3. Single-clip test

One clip through the encoder, to check the output shape, GPU memory and time per clip before processing the whole dataset.

In [3]:
clip = splits.iloc[0]
frames = read_clip(DATA_DIR / clip["clip"])              # (16, 360, 640, 3), uint8
video = torch.from_numpy(frames).permute(0, 3, 1, 2)     # (T, C, H, W): the layout the processor expects
inputs = processor(video, return_tensors="pt").to(device=device, dtype=torch.float16)
print(clip["clip"], "|", {k: tuple(v.shape) for k, v in inputs.items()})

for run in range(2):   # the first run includes one-time GPU warm-up
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    with torch.inference_mode():
        tokens = model.get_vision_features(**inputs)
    torch.cuda.synchronize()
    print(f"run {run + 1}: {time.perf_counter() - start:.2f}s | "
          f"peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB | tokens {tuple(tokens.shape)}")

test/360/360.mp4 | {'pixel_values_videos': (1, 16, 3, 256, 256)}
run 1: 2.66s | peak GPU memory 0.72 GB | tokens (1, 2048, 1024)
run 2: 0.22s | peak GPU memory 0.72 GB | tokens (1, 2048, 1024)


## 4. Feature extraction

The encoder returns 2048 tokens per clip: 8 time slices x 256 spatial patches (16 x 16), 1024 dimensions each.
We keep the average over space for each time slice, so every clip becomes an (8, 1024) array.
This keeps the temporal order, which distinguishes maneuvers, at 1/256 of the storage.

In [4]:
FEATURES_FILE = PROJECT_ROOT / "data" / "processed" / "vjepa2_vitl_t8.npz"

def encode_clip(path):
    """Encoder tokens of one clip averaged over space: (8, 1024), one vector per time slice."""
    frames = read_clip(path)
    video = torch.from_numpy(frames).permute(0, 3, 1, 2)
    inputs = processor(video, return_tensors="pt").to(device=device, dtype=torch.float16)
    with torch.inference_mode():
        tokens = model.get_vision_features(**inputs)          # (1, 2048, 1024)
    # Tokens are ordered by time, then space: 8 time slices x 256 patches
    return tokens.reshape(8, 256, -1).mean(dim=1).float().cpu().numpy()

features = encode_clip(DATA_DIR / clip["clip"])
features.shape

(8, 1024)

In [5]:
def save_features(done):
    """Save all extracted features, with the clip list to keep rows aligned."""
    FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
    clips = list(done)
    np.savez(FEATURES_FILE, clips=np.array(clips), features=np.stack([done[c] for c in clips]))

# Resume from a previous partial run, if any
done = {}
if FEATURES_FILE.exists():
    saved = np.load(FEATURES_FILE)
    done = dict(zip(saved["clips"], saved["features"]))

todo = [c for c in splits["clip"] if c not in done]
print(f"{len(done)} clips already done, {len(todo)} to go")

start = time.perf_counter()
for k, rel_path in enumerate(todo, 1):
    done[rel_path] = encode_clip(DATA_DIR / rel_path)
    if k % 100 == 0 or k == len(todo):
        save_features(done)
        print(f"{len(done)}/{len(splits)} clips | {time.perf_counter() - start:.0f}s", flush=True)

0 clips already done, 1056 to go
100/1056 clips | 41s
200/1056 clips | 78s
300/1056 clips | 116s
400/1056 clips | 154s
500/1056 clips | 192s
600/1056 clips | 230s
700/1056 clips | 267s
800/1056 clips | 303s
900/1056 clips | 340s
1000/1056 clips | 379s
1056/1056 clips | 399s


## 5. Summary

- Encoder: V-JEPA 2 ViT-L (326M parameters), float16, on an RTX 3050 Laptop GPU (6 GB).
- One clip: 0.22 s on the GPU, 0.72 GB peak memory, 2048 tokens of 1024 dimensions.
- All 1056 clips: 6.6 minutes including video decoding (about 0.38 s per clip).
- Output: `data/processed/vjepa2_vitl_t8.npz`, one (8, 1024) array per clip (average over space for each time slice). Not versioned; rebuild it with this notebook.